# Joan Tryhard

### Imports

In [137]:
import pandas as pd
import sklearn
import imblearn
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix



### Get Data and Preprocess

In [138]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd

TESTING_WITH_TRAIN_DATA = False
TESTING_WITH_KAGGLE_DATA = True


train = pd.read_csv("data/train_dataset_processed.csv")
test = pd.read_csv("data/test_dataset_processed.csv")

languages = train['language'].unique()


if TESTING_WITH_TRAIN_DATA:

    # Split into train and test sets
    sentences = train[['language', 'sentence_id']].drop_duplicates()
    sentences_train, sentences_test = train_test_split(sentences,test_size=0.2,random_state=42)
    train_set = pd.merge(train, sentences_train, on=['language', 'sentence_id'])
    test_set = pd.merge(train, sentences_test, on=['language', 'sentence_id'])

    # One-hot encode 'language' in train and test sets with language_LANGUAGE
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train_set[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train_set.index)
    train = pd.concat([train_set.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test_set[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test_set.index)
    test = pd.concat([test_set.drop(columns=['language']), language_df_test], axis=1)

    # Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test.drop(columns=['root'])
    y_test = test['root']

else: 
    # One-hot encode 'language' in train and test (all test data) sets
    enc = OneHotEncoder(sparse_output=True)
    language_encoded = enc.fit_transform(train[['language']])
    language_df = pd.DataFrame(language_encoded.toarray(),
                            columns=enc.get_feature_names_out(['language']),
                            index=train.index)
    train = pd.concat([train.drop(columns=['language']), language_df], axis=1)

    # Same for test set
    language_encoded_test = enc.transform(test[['language']])
    language_df_test = pd.DataFrame(language_encoded_test.toarray(),
                                    columns=enc.get_feature_names_out(['language']),
                                    index=test.index)
    test = pd.concat([test.drop(columns=['language']), language_df_test], axis=1)

    #Prepare data and labels
    X_train = train.drop(columns=['root'])
    y_train = train['root']
    X_test = test
    # there is no y_test, it is what we want to predict!

## Models

### Unimodel Random Forest

In [139]:
from sklearn.ensemble import RandomForestClassifier
# import linear classifier
clf = RandomForestClassifier(class_weight='balanced', random_state = 42)
clf.fit(X_train, y_train)

prob_predictions = clf.predict_proba(X_test)
# convert to 0s and 1s integers
predictions = []
for i in range(len(prob_predictions)):
    if prob_predictions[i][1] > 0.5:
        predictions.append(1)
    else:
        predictions.append(0)

### Multimodel Random Forest

In [140]:
# Step 1: Detect one-hot language columns
language_cols = [col for col in X_train.columns if col.startswith('language_')]

# Step 2: Store models per language
models = {}

# Step 3: Train one model per language
for lang_col in language_cols:
    lang_X_train = X_train[X_train[lang_col] == 1]
    lang_y_train = y_train[lang_X_train.index]
    
    clf = RandomForestClassifier(class_weight='balanced', random_state=42)
    clf.fit(lang_X_train.drop(columns=language_cols), lang_y_train)
    models[lang_col] = clf

# Step 4: Preserve original order in test set
X_test_ordered = X_test.copy()
X_test_ordered['original_index'] = X_test_ordered.index

# Step 5: Predict for each language group
predictions = []

for lang_col in language_cols:
    lang_X_test = X_test_ordered[X_test_ordered[lang_col] == 1].copy()
    lang_X_test_features = lang_X_test.drop(columns=language_cols + ['original_index'])
    
    if not lang_X_test_features.empty:
        clf = models[lang_col]
        probs = clf.predict_proba(lang_X_test_features)
        lang_predictions = (probs[:, 1] > 0.5).astype(int)
    else:
        lang_predictions = np.array([], dtype=int)

    lang_results = pd.DataFrame({
        'original_index': lang_X_test['original_index'].values,
        'prediction': lang_predictions
    })
    predictions.append(lang_results)

# Step 6: Combine and sort all predictions to match original test set order
all_predictions = pd.concat(predictions).sort_values('original_index')

# Final list of predictions
final_predictions = all_predictions['prediction'].tolist()

## Evaluating results

In [141]:
if TESTING_WITH_TRAIN_DATA:

    y_test_labels = y_test.values

    # Get the error on the test set with different metrics
    print(confusion_matrix(y_test_labels, predictions))
    print(classification_report(y_test_labels, predictions, target_names=['0', '1']))

In [142]:
def find_root(group):
    # Check if there's a row with root == 1
    root_rows = group[group['root'] == 1]
    if not root_rows.empty:
        return root_rows.iloc[0]['node']
    else:
        return 1

if not TESTING_WITH_TRAIN_DATA:
    # add predictions to the test set
    X_test['root'] = predictions

    # Create a language column with the original language values, which are now one-hot encoded
    language_columns = [col for col in X_test.columns if col.startswith('language_')]
    language_values = enc.inverse_transform(X_test[language_columns])[:, 0]
    X_test['language'] = language_values
    # Drop the one-hot encoded language columns
    X_test = X_test.drop(columns=language_columns)

    # Group by language and sentence_id
    grouped = X_test.groupby(['language', 'sentence_id'])

    # Apply the root finding function to each group
    roots = grouped.apply(find_root).reset_index(name='root')

    # Add a sequential id column
    roots.insert(0, 'id', range(1, len(roots) + 1))
    roots = roots[['id', 'root']]

    # Save to CSV
    roots.to_csv('data/predictions_submission.csv', index=False)

    # # For each pair of language, sentence_id, iterate over rows with that pair to create just a row containing index, row[node] of the row with a 1 in the root column, 
    # # and if there is no 1 in the root column, create a row with index, 0
    # index = 1
    # rows = []
    # unique_pairs = X_test[['language', 'sentence_id']].drop_duplicates().values
    # for language, sentence_id in tqdm(unique_pairs, desc="Creating submission rows"):
    #     # Get the rows with that language and sentence_id
    #     rows_with_language_and_sentence_id = X_test[(X_test['language'] == language) & (X_test['sentence_id'] == sentence_id)]
    #     # Check if there is a 1 in the root column
    #     if 1 in rows_with_language_and_sentence_id['root'].values:
    #         # Get the index of the row with a 1 in the root column
    #         index_of_1 = rows_with_language_and_sentence_id[rows_with_language_and_sentence_id['root'] == 1].index[0]
    #         # Create a row with index, row[node] of the row with a 1 in the root column
    #         rows.append([index, rows_with_language_and_sentence_id.loc[index_of_1, 'node']])
    #     else:
    #         # Create a row with index, 0
    #         rows.append([index, 1])
    #     index += 1
    # # Create a dataframe with the rows
    # df = pd.DataFrame(rows, columns=['id', 'root'])
    # df.to_csv('data/predictions_submission.csv', index=False)

current_predictions = pd.read_csv('data/predictions_submission.csv')
current_predictions.head()

ValueError: Length of values (21) does not match length of index (194648)

In [ ]:
if TESTING_WITH_KAGGLE_DATA:
    kaggle_perfect_predictions = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions = pd.read_csv('data/predictions_submission.csv')
    kaggle_perfect_predictions = kaggle_perfect_predictions.drop(columns=['id'])
    current_predictions = current_predictions.drop(columns=['id'])
    current_predictions = current_predictions.drop(index=0)
    kaggle_perfect_predictions = kaggle_perfect_predictions.drop(index=0)
    # Get the error on the kaggle test set 


print(classification_report(kaggle_perfect_predictions, current_predictions))

              precision    recall  f1-score   support

           1       0.06      0.90      0.12       690
           2       0.14      0.01      0.02       675
           3       0.19      0.02      0.04       687
           4       0.09      0.01      0.02       640
           5       0.11      0.02      0.03       693
           6       0.17      0.01      0.03       626
           7       0.04      0.00      0.01       653
           8       0.10      0.01      0.01       607
           9       0.08      0.00      0.01       547
          10       0.11      0.00      0.01       510
          11       0.00      0.00      0.00       491
          12       0.05      0.00      0.00       407
          13       0.00      0.00      0.00       390
          14       0.00      0.00      0.00       346
          15       0.00      0.00      0.00       336
          16       0.11      0.00      0.01       312
          17       0.00      0.00      0.00       248
          18       0.00    

/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/joan/Master-Data-Science-FIB-UPC/2nd semester/ML- Machine Learning/ML-Machine-Learning/PROJECT/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: 